# REWARD MEAN + LOSS MEAN + TIMESTEPS MEAN

In [16]:
# -*- coding: utf-8 -*-
"""
merged_tb_plots.py
Une os dois scripts: gera (por algoritmo)
  1) Reward (tag auto-escolhida por prioridade/regex)
  2) Episode length (episode_len_mean)
  3) Policy losses (total_loss de policy_0..4 no mesmo gráfico)

Saída: <base_dir>/charts/<algo>/*.png

Uso no terminal:
  python merged_tb_plots.py --base-dir ./exp_results --all

Uso em Jupyter/Notebook:
  # apenas execute a célula; parse_known_args() ignora argumentos do kernel
"""

import os
import re
import glob
import argparse
from collections import defaultdict, Counter
from typing import Optional, Set, Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.python.summary.summary_iterator import summary_iterator
from scipy.ndimage import gaussian_filter1d

# ------------------ leitura de eventos ------------------
def get_event_files_in_dir(d: str) -> List[str]:
    return sorted(glob.glob(os.path.join(d, "**", "events.out.tfevents.*"), recursive=True))

def safe_iter_events(path: str):
    try:
        for e in summary_iterator(path):
            yield e
    except Exception as ex:
        print(f"⚠️  Ignorando {path} (erro de leitura: {ex})")

# ------------------ utilitários de tags ------------------
RAY_PREFIX = re.compile(r"^(ray/(tune|train|rllib)/)")
def strip_ray_prefix(tag: str) -> str:
    # remove "ray/tune/" ou "ray/train/" etc. para facilitar matching
    return RAY_PREFIX.sub("", tag)

def tag_frequency(event_files, max_per_file=500) -> Tuple[Counter, Counter]:
    """Conta tags distintas (uma vez por arquivo) e também a versão 'stripped' sem prefixos do Ray."""
    raw = Counter()
    stripped = Counter()
    for path in event_files:
        seen_raw = set()
        seen_stripped = set()
        steps_seen = 0
        for e in safe_iter_events(path):
            for v in e.summary.value:
                t = v.tag
                ts = strip_ray_prefix(t)
                if t not in seen_raw:
                    raw[t] += 1; seen_raw.add(t)
                if ts not in seen_stripped:
                    stripped[ts] += 1; seen_stripped.add(ts)
            steps_seen += 1
            if steps_seen >= max_per_file:
                break
    return raw, stripped

# Reward (forma "stripped")
REWARD_TAG_PREFERENCE = [
    "evaluation/episode_reward_mean",
    "episode_reward_mean",
    "rollout/episode_reward_mean",
    "episode_return_mean",
    "evaluation/episode_return_mean",
    "train/episode_reward_mean",
    "train/mean_reward",
    "charts/episode_reward_mean",
]
REWARD_REGEX = re.compile(
    r"(episode|ep).*(reward|return).*(mean|avg)|"
    r"(reward|return).*(episode|ep).*(mean|avg)",
    re.IGNORECASE,
)

# Eixo X (forma "stripped")
X_AXIS_TAG_PREFERENCE = [
    "timesteps_total",
    "env_steps_sampled",
    "env_steps_trained",
    "training_iteration",
    "episodes_total",
]

def choose_reward_tag(available_stripped_tags: Set[str]) -> Optional[str]:
    for t in REWARD_TAG_PREFERENCE:
        if t in available_stripped_tags:
            return t
    candidates = [t for t in available_stripped_tags if REWARD_REGEX.search(t)]
    if candidates:
        candidates.sort(key=lambda s: (len(s), s))
        return candidates[0]
    return None

def choose_x_axis_tag(available_stripped_tags: Set[str]) -> Optional[str]:
    for t in X_AXIS_TAG_PREFERENCE:
        if t in available_stripped_tags:
            return t
    return None  # fallback: step do TensorBoard

# ------------------ extração ------------------
def extract_series(event_paths, y_tag, x_tag=None):
    """
    Agrega um escalar y_tag em múltiplas execuções.
    Se x_tag for fornecida, alinha usando o 'step' do TF e retorna x a partir dessa tag; senão usa 'step'.
    Retorna (xs_sorted, mean_y, std_y).
    As tags devem ser fornecidas em forma *stripped* (sem prefixo ray/*),
    mas o matching aceita ambas as formas.
    """
    def matches(tag, wanted):
        return strip_ray_prefix(tag) == wanted

    x_by_step_all_runs = defaultdict(list) if x_tag else None
    y_by_step = defaultdict(list)

    for path in event_paths:
        x_by_step = {}       # step -> x
        y_by_step_run = {}   # step -> y
        for e in safe_iter_events(path):
            for v in e.summary.value:
                t = v.tag
                if x_tag and matches(t, x_tag):
                    x_by_step[e.step] = float(v.simple_value)
                if matches(t, y_tag):
                    y_by_step_run[e.step] = float(v.simple_value)
        if x_tag:
            for s, xv in x_by_step.items():
                x_by_step_all_runs[s].append(xv)
        for s, yv in y_by_step_run.items():
            y_by_step[s].append(yv)

    steps = sorted(y_by_step.keys())
    if not steps:
        return [], [], []
    if x_tag:
        xs = [np.mean(x_by_step_all_runs[s]) if s in x_by_step_all_runs else float(s) for s in steps]
    else:
        xs = [float(s) for s in steps]

    means = [np.mean(y_by_step[s]) for s in steps]
    stds  = [np.std(y_by_step[s])  for s in steps]
    order = np.argsort(xs)
    xs = list(np.array(xs)[order])
    means = list(np.array(means)[order])
    stds = list(np.array(stds)[order])
    return xs, means, stds

# ------------------ plotagem ------------------
def plot_smoothed_line(xs, means, stds, label="metric", title="", normalize=False, sigma=3, save_path=None, ylabel="Value"):
    plt.figure(figsize=(11, 7))
    means = np.array(means); stds = np.array(stds); xs = np.array(xs)
    # Suaviza só se houver pontos suficientes
    sm_m = gaussian_filter1d(means, sigma=sigma) if len(means) >= 3 else means
    sm_s = gaussian_filter1d(stds,  sigma=sigma) if len(stds)  >= 3 else stds
    if normalize:
        mn, mx = sm_m.min(), sm_m.max()
        rng = (mx - mn) if mx > mn else 1.0
        sm_m = (sm_m - mn) / rng
        sm_s = sm_s / rng
        lower = np.clip(sm_m - sm_s, 0, 1); upper = np.clip(sm_m + sm_s, 0, 1)
        plt.ylabel("Normalized value")
    else:
        lower, upper = sm_m - sm_s, sm_m + sm_s
        plt.ylabel(ylabel)
    plt.plot(xs, sm_m, label=label)
    plt.fill_between(xs, lower, upper, alpha=0.2)
    xlabel = "Timesteps"
    if not len(xs) or (xs[0] == 0 and max(xs, default=0) <= len(xs)):
        xlabel = "Step"
    plt.xlabel(xlabel)
    plt.title(title, fontsize=16, fontweight="bold")
    plt.legend(); plt.grid(True); plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=160)
        print(f"💾 Salvo: {save_path}")
        plt.close()
    else:
        plt.show()

def plot_multi_smoothed(series_dict, title="", normalize=False, sigma=3, save_path=None, ylabel="Value"):
    """
    series_dict: {label: (xs, means, stds)}
    Plota múltiplas séries (mesmo eixo X aproximado).
    """
    plt.figure(figsize=(11, 7))
    xlabel = "Timesteps"
    any_x = None

    for label, (xs, means, stds) in series_dict.items():
        if not xs:
            continue
        any_x = xs if any_x is None else any_x
        means = np.array(means); stds = np.array(stds); xs = np.array(xs)
        sm_m = gaussian_filter1d(means, sigma=sigma) if len(means) >= 3 else means
        sm_s = gaussian_filter1d(stds,  sigma=sigma) if len(stds)  >= 3 else stds
        if normalize:
            mn, mx = sm_m.min(), sm_m.max()
            rng = (mx - mn) if mx > mn else 1.0
            sm_m = (sm_m - mn) / rng
            sm_s = sm_s / rng
            lower = np.clip(sm_m - sm_s, 0, 1); upper = np.clip(sm_m + sm_s, 0, 1)
            plt.ylabel("Normalized value")
        else:
            lower, upper = sm_m - sm_s, sm_m + sm_s
            plt.ylabel(ylabel)
        plt.plot(xs, sm_m, label=label)
        plt.fill_between(xs, lower, upper, alpha=0.15)

    if any_x:
        if any_x[0] == 0 and max(any_x, default=0) <= len(any_x):
            xlabel = "Step"
    plt.xlabel(xlabel)
    plt.title(title, fontsize=16, fontweight="bold")
    plt.legend(); plt.grid(True); plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=160)
        print(f"💾 Salvo: {save_path}")
        plt.close()
    else:
        plt.show()

# ------------------ agrupar por algoritmo ------------------
def infer_algorithm(dirname: str) -> str:
    """
    Obtém o nome do algoritmo pelo prefixo antes do primeiro '_' no nome da pasta.
    Ex.: 'happo_mlp_sunt_bus_...' -> 'happo'
    """
    base = os.path.basename(dirname).lower()
    if "_" in base:
        return base.split("_", 1)[0]
    return base  # fallback: a pasta inteira

def group_folders_by_algorithm(base_dir: str) -> Dict[str, List[str]]:
    """Agrupa subpastas imediatas por algoritmo."""
    groups = defaultdict(list)
    for d in os.listdir(base_dir):
        full = os.path.join(base_dir, d)
        if os.path.isdir(full):
            algo = infer_algorithm(d)
            groups[algo].append(full)
    # Se houver events diretamente no base_dir, trate como pasta do próprio nome do base_dir
    root_events = get_event_files_in_dir(base_dir)
    if root_events:
        algo = infer_algorithm(os.path.basename(base_dir))
        groups[algo].append(base_dir)
    return groups

# ------------------ métricas alvo (forma 'stripped') ------------------
EPISODE_LEN_TAG = "episode_len_mean"
POLICY_LOSS_TAGS = [f"info/learner/policy_{k}/learner_stats/total_loss" for k in range(5)]

# ------------------ driver ------------------
def main():
    ap = argparse.ArgumentParser(description="Gera gráficos a partir de TensorBoard (agrupado por algoritmo).")
    ap.add_argument("--base-dir", type=str, default="./exp_results", help="Diretório base com pastas/runs.")
    ap.add_argument("--charts-dir", type=str, default=None, help="Diretório de saída (default: <base-dir>/charts)")
    ap.add_argument("--all", action="store_true", help="Gera todos os gráficos (reward + ep_len + policy_loss).")
    ap.add_argument("--reward", dest="reward", action="store_true", help="Gera reward (auto tag).")
    ap.add_argument("--no-reward", dest="reward", action="store_false", help="Não gera reward.")
    ap.add_argument("--ep-len", dest="ep_len", action="store_true", help="Gera episode_len_mean.")
    ap.add_argument("--no-ep-len", dest="ep_len", action="store_false", help="Não gera episode_len_mean.")
    ap.add_argument("--policy-loss", dest="policy_loss", action="store_true", help="Gera policy total_loss (0..4) em um gráfico.")
    ap.add_argument("--no-policy-loss", dest="policy_loss", action="store_false", help="Não gera policy total_loss.")
    ap.add_argument("--sigma", type=float, default=3.0, help="Sigma do Gaussian smoothing.")
    ap.add_argument("--normalize", action="store_true", help="Normaliza curvas para [0,1].")
    ap.set_defaults(reward=None, ep_len=None, policy_loss=None)

    # <<< importante: ignora args desconhecidos do Jupyter/IPython >>>
    args, _unknown = ap.parse_known_args()

    base_dir = args.base_dir
    charts_dir = args.charts_dir or os.path.join(base_dir, "charts")
    os.makedirs(charts_dir, exist_ok=True)

    # resolver flags
    if args.all:
        reward = True; ep_len = True; policy_loss = True
    else:
        # default = tudo se nada for especificado
        reward = True if args.reward is None else args.reward
        ep_len = True if args.ep_len is None else args.ep_len
        policy_loss = True if args.policy_loss is None else args.policy_loss

    # Agrupar pastas por algoritmo
    algo_groups = group_folders_by_algorithm(base_dir)

    for algo, folders in sorted(algo_groups.items()):
        # Junta todos os event files de TODAS as pastas desse algoritmo
        event_files = []
        for f in folders:
            event_files.extend(get_event_files_in_dir(f))
        event_files = sorted(set(event_files))
        if not event_files:
            print(f"… {algo}: sem arquivos de eventos.")
            continue

        # Descobre tags disponíveis (stripped)
        _, stripped_freq = tag_frequency(event_files, max_per_file=800)
        available = set(stripped_freq.keys())
        x_tag = choose_x_axis_tag(available)

        # --- reward (auto) ---
        if reward:
            y_tag = choose_reward_tag(available)
            if y_tag:
                xs, means, stds = extract_series(event_files, y_tag=y_tag, x_tag=x_tag)
                if xs:
                    title = f"{algo.upper()} — {y_tag}" + (f" vs {x_tag}" if x_tag else "")
                    fn = f"{algo}/{algo}__{y_tag.replace('/','_')}" + (f"__{x_tag.replace('/','_')}" if x_tag else "__step") + ".png"
                    save_path = os.path.join(charts_dir, fn)
                    plot_smoothed_line(
                        xs, means, stds,
                        label=y_tag,
                        title=title,
                        normalize=args.normalize,
                        sigma=args.sigma,
                        save_path=save_path,
                        ylabel="Reward"
                    )
                else:
                    print(f"⚠️  {algo}: sem dados para '{y_tag}'.")
            else:
                print(f"… {algo}: nenhuma tag de reward identificada.")

        # --- episode_len_mean ---
        if ep_len:
            if EPISODE_LEN_TAG in available:
                xs, means, stds = extract_series(event_files, y_tag=EPISODE_LEN_TAG, x_tag=x_tag)
                if xs:
                    title = f"{algo.upper()} — {EPISODE_LEN_TAG}" + (f" vs {x_tag}" if x_tag else "")
                    fn = f"{algo}/{algo}__{EPISODE_LEN_TAG.replace('/','_')}" + (f"__{x_tag.replace('/','_')}" if x_tag else "__step") + ".png"
                    save_path = os.path.join(charts_dir, fn)
                    plot_smoothed_line(
                        xs, means, stds,
                        label=EPISODE_LEN_TAG,
                        title=title,
                        normalize=args.normalize,
                        sigma=args.sigma,
                        save_path=save_path,
                        ylabel="Episode length"
                    )
                else:
                    print(f"⚠️  {algo}: sem dados para '{EPISODE_LEN_TAG}'.")
            else:
                print(f"… {algo}: tag '{EPISODE_LEN_TAG}' não encontrada.")

        # --- policy losses (multi-plot) ---
        if policy_loss:
            series = {}
            present_losses = [t for t in POLICY_LOSS_TAGS if t in available]
            if not present_losses:
                print(f"… {algo}: nenhuma policy loss encontrada (policy_0..4).")
            else:
                for t in present_losses:
                    xs, means, stds = extract_series(event_files, y_tag=t, x_tag=x_tag)
                    if xs:
                        # label curto: policy_k
                        try:
                            lbl = t.split('/')[2]  # policy_k
                        except Exception:
                            lbl = t
                        series[lbl] = (xs, means, stds)
                if series:
                    title = f"{algo.upper()} — learner_stats/total_loss por policy" + (f" vs {x_tag}" if x_tag else "")
                    fn = f"{algo}/{algo}__policies_total_loss" + (f"__{x_tag.replace('/','_')}" if x_tag else "__step") + ".png"
                    save_path = os.path.join(charts_dir, fn)
                    plot_multi_smoothed(series, title=title, normalize=args.normalize, sigma=args.sigma, save_path=save_path, ylabel="Total loss")
                else:
                    print(f"⚠️  {algo}: sem dados válidos para total_loss das policies.")

    print("✅ Concluído.")

if __name__ == "__main__":
    main()


… charts: sem arquivos de eventos.
💾 Salvo: ./exp_results/charts/coma/coma__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results/charts/coma/coma__episode_len_mean__timesteps_total.png
… coma: nenhuma policy loss encontrada (policy_0..4).
💾 Salvo: ./exp_results/charts/exp/exp__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results/charts/exp/exp__episode_len_mean__timesteps_total.png
💾 Salvo: ./exp_results/charts/exp/exp__policies_total_loss__timesteps_total.png
💾 Salvo: ./exp_results/charts/happo/happo__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results/charts/happo/happo__episode_len_mean__timesteps_total.png
💾 Salvo: ./exp_results/charts/happo/happo__policies_total_loss__timesteps_total.png
💾 Salvo: ./exp_results/charts/hatrpo/hatrpo__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results/charts/hatrpo/hatrpo__episode_len_mean__timesteps_total.png
💾 Salvo: ./exp_results/charts/hatrpo/hatrpo__policies_total_loss__timesteps_total.png
💾 Salvo: 

# Gráfico normalizado

In [18]:
# -*- coding: utf-8 -*-
"""
reward_fixed_norm_plots.py
Gera, por algoritmo, o gráfico de REWARD normalizado em [0,1] com mapeamento linear fixo:
    norm = clip( (reward - NORM_MIN) / (NORM_MAX - NORM_MIN), 0, 1 )
por padrão: NORM_MIN = -650  -> 0
            NORM_MAX =  100  -> 1

Agrega múltiplas execuções por algoritmo a partir de eventos do TensorBoard
e aplica suavização Gaussiana antes da normalização (configurável).

>>> O eixo Y é SEMPRE fixado em [0, 1]. <<<

Saída: <base_dir>/charts/<algo>/<algo>__<reward_tag>__<x_tag|step>__norm_fixed.png

Uso (terminal):
  python reward_fixed_norm_plots.py --base-dir ./exp_results_copy --sigma 3
"""

import os
import re
import glob
import argparse
from collections import defaultdict, Counter
from typing import Optional, Set, Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.python.summary.summary_iterator import summary_iterator
from scipy.ndimage import gaussian_filter1d

# ------------ leitura de eventos ------------
def get_event_files_in_dir(d: str) -> List[str]:
    return sorted(glob.glob(os.path.join(d, "**", "events.out.tfevents.*"), recursive=True))

def safe_iter_events(path: str):
    try:
        for e in summary_iterator(path):
            yield e
    except Exception as ex:
        print(f"⚠️  Ignorando {path} (erro de leitura: {ex})")

# ------------ utilitários de tags ------------
RAY_PREFIX = re.compile(r"^(ray/(tune|train|rllib)/)")
def strip_ray_prefix(tag: str) -> str:
    return RAY_PREFIX.sub("", tag)

def tag_frequency(event_files, max_per_file=500) -> Tuple[Counter, Counter]:
    raw, stripped = Counter(), Counter()
    for path in event_files:
        seen_raw, seen_stripped = set(), set()
        steps_seen = 0
        for e in safe_iter_events(path):
            for v in e.summary.value:
                t = v.tag
                ts = strip_ray_prefix(t)
                if t not in seen_raw:
                    raw[t] += 1; seen_raw.add(t)
                if ts not in seen_stripped:
                    stripped[ts] += 1; seen_stripped.add(ts)
            steps_seen += 1
            if steps_seen >= max_per_file:
                break
    return raw, stripped

# Escolha automática da tag de reward (forma "stripped")
REWARD_TAG_PREFERENCE = [
    "evaluation/episode_reward_mean",
    "episode_reward_mean",
    "rollout/episode_reward_mean",
    "episode_return_mean",
    "evaluation/episode_return_mean",
    "train/episode_reward_mean",
    "train/mean_reward",
    "charts/episode_reward_mean",
]
REWARD_REGEX = re.compile(
    r"(episode|ep).*(reward|return).*(mean|avg)|"
    r"(reward|return).*(episode|ep).*(mean|avg)",
    re.IGNORECASE,
)

X_AXIS_TAG_PREFERENCE = [
    "timesteps_total",
    "env_steps_sampled",
    "env_steps_trained",
    "training_iteration",
    "episodes_total",
]

def choose_reward_tag(available: Set[str]) -> Optional[str]:
    for t in REWARD_TAG_PREFERENCE:
        if t in available:
            return t
    candidates = [t for t in available if REWARD_REGEX.search(t)]
    if candidates:
        candidates.sort(key=lambda s: (len(s), s))
        return candidates[0]
    return None

def choose_x_axis_tag(available: Set[str]) -> Optional[str]:
    for t in X_AXIS_TAG_PREFERENCE:
        if t in available:
            return t
    return None

# ------------ extração ------------
def extract_series(event_paths, y_tag, x_tag=None):
    """
    Retorna xs, mean_y, std_y agregando por step entre múltiplos runs.
    Matching aceita tag com/sem prefixo ray/* (compara versão 'stripped').
    """
    def matches(tag, wanted):
        return strip_ray_prefix(tag) == wanted

    x_by_step_all_runs = defaultdict(list) if x_tag else None
    y_by_step = defaultdict(list)

    for path in event_paths:
        x_by_step = {}
        y_by_step_run = {}
        for e in safe_iter_events(path):
            for v in e.summary.value:
                t = v.tag
                if x_tag and matches(t, x_tag):
                    x_by_step[e.step] = float(v.simple_value)
                if matches(t, y_tag):
                    y_by_step_run[e.step] = float(v.simple_value)
        if x_tag:
            for s, xv in x_by_step.items():
                x_by_step_all_runs[s].append(xv)
        for s, yv in y_by_step_run.items():
            y_by_step[s].append(yv)

    steps = sorted(y_by_step.keys())
    if not steps:
        return [], [], []
    if x_tag:
        xs = [np.mean(x_by_step_all_runs[s]) if s in x_by_step_all_runs else float(s) for s in steps]
    else:
        xs = [float(s) for s in steps]

    means = [np.mean(y_by_step[s]) for s in steps]
    stds  = [np.std(y_by_step[s])  for s in steps]
    order = np.argsort(xs)
    xs = list(np.array(xs)[order])
    means = list(np.array(means)[order])
    stds = list(np.array(stds)[order])
    return xs, means, stds

# ------------ normalização fixa ------------
def fixed_linear_norm(values: np.ndarray, vmin: float, vmax: float) -> np.ndarray:
    """Aplica normalização linear fixa e clamp para [0,1]."""
    denom = (vmax - vmin) if vmax != vmin else 1.0
    norm = (values - vmin) / denom
    return np.clip(norm, 0.0, 1.0)

# ------------ plotagem ------------
def plot_smoothed_fixed_norm(xs, means, stds, label, title, sigma, norm_min, norm_max, save_path=None):
    """
    1) Suaviza média e desvio (gaussian)
    2) Normaliza (0..1) com mapeamento fixo: -650->0 ; 100->1 (ou conforme flags)
    3) Mantém o Y SEMPRE em [0,1]
    """
    plt.figure(figsize=(11,7))
    xs = np.array(xs); m = np.array(means); s = np.array(stds)

    sm_m = gaussian_filter1d(m, sigma=sigma) if len(m) >= 3 else m
    sm_s = gaussian_filter1d(s, sigma=sigma) if len(s) >= 3 else s

    nm = fixed_linear_norm(sm_m, norm_min, norm_max)
    # escala o desvio na mesma razão (linear) e clampa
    denom = (norm_max - norm_min) if norm_max != norm_min else 1.0
    ns = np.clip(sm_s / denom, 0.0, 1.0)

    lower = np.clip(nm - ns, 0, 1)
    upper = np.clip(nm + ns, 0, 1)

    plt.plot(xs, nm, label=label)
    plt.fill_between(xs, lower, upper, alpha=0.2)

    # EIXO X
    xlabel = "Timesteps"
    if not len(xs) or (len(xs) and xs[0] == 0 and max(xs, default=0) <= len(xs)):
        xlabel = "Step"
    plt.xlabel(xlabel)

    # EIXO Y FIXO [0,1]
    plt.ylabel(f"Normalized reward (0–1)  [{norm_min}→0, {norm_max}→1]")
    plt.ylim(0, 1)
    plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])

    plt.title(title, fontsize=16, fontweight="bold")
    plt.legend(); plt.grid(True); plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=160)
        print(f"💾 Salvo: {save_path}")
        plt.close()
    else:
        plt.show()

# ------------ agrupamento por algoritmo ------------
def infer_algorithm(dirname: str) -> str:
    base = os.path.basename(dirname).lower()
    if "_" in base:
        return base.split("_", 1)[0]
    return base

def group_folders_by_algorithm(base_dir: str) -> Dict[str, List[str]]:
    groups = defaultdict(list)
    for d in os.listdir(base_dir):
        full = os.path.join(base_dir, d)
        if os.path.isdir(full):
            algo = infer_algorithm(d)
            groups[algo].append(full)
    root_events = get_event_files_in_dir(base_dir)
    if root_events:
        algo = infer_algorithm(os.path.basename(base_dir))
        groups[algo].append(base_dir)
    return groups

# ------------ driver ------------
def main():
    ap = argparse.ArgumentParser(description="Reward normalizado (0–1) com mapeamento fixo por algoritmo.")
    ap.add_argument("--base-dir", type=str, default="./exp_results", help="Diretório base com pastas/runs.")
    ap.add_argument("--charts-dir", type=str, default=None, help="Saída (default: <base-dir>/charts)")
    ap.add_argument("--sigma", type=float, default=3.0, help="Sigma da suavização Gaussian.")
    ap.add_argument("--norm-min", type=float, default=-650.0, help="Valor do reward que será mapeado para 0.")
    ap.add_argument("--norm-max", type=float, default=100.0, help="Valor do reward que será mapeado para 1.")
    ap.add_argument("--max-tags-scan", type=int, default=800, help="Eventos por arquivo ao vasculhar tags.")
    # compatível com Jupyter/IPython
    args, _unknown = ap.parse_known_args()

    base_dir   = args.base_dir
    charts_dir = args.charts_dir or os.path.join(base_dir, "charts")
    os.makedirs(charts_dir, exist_ok=True)

    algo_groups = group_folders_by_algorithm(base_dir)

    for algo, folders in sorted(algo_groups.items()):
        event_files = []
        for f in folders:
            event_files.extend(get_event_files_in_dir(f))
        event_files = sorted(set(event_files))
        if not event_files:
            print(f"… {algo}: sem arquivos de eventos.")
            continue

        # tags disponíveis
        _, stripped = tag_frequency(event_files, max_per_file=args.max_tags_scan)
        available = set(stripped.keys())
        x_tag = choose_x_axis_tag(available)
        y_tag = choose_reward_tag(available)
        if not y_tag:
            print(f"… {algo}: nenhuma tag de reward identificada.")
            continue

        xs, means, stds = extract_series(event_files, y_tag=y_tag, x_tag=x_tag)
        if not xs:
            print(f"⚠️  {algo}: sem dados para '{y_tag}'.")
            continue

        title = f"{algo.upper()} — {y_tag} (fixed normalization)" + (f" vs {x_tag}" if x_tag else "")
        fn = f"{algo}/{algo}__{y_tag.replace('/','_')}__{x_tag.replace('/','_') if x_tag else 'step'}__norm_fixed.png"
        save_path = os.path.join(charts_dir, fn)

        plot_smoothed_fixed_norm(
            xs, means, stds,
            label=y_tag,
            title=title,
            sigma=args.sigma,
            norm_min=args.norm_min,
            norm_max=args.norm_max,
            save_path=save_path
        )

    print("✅ Concluído.")

if __name__ == "__main__":
    main()


… charts: sem arquivos de eventos.
💾 Salvo: ./exp_results/charts/coma/coma__episode_reward_mean__timesteps_total__norm_fixed.png
💾 Salvo: ./exp_results/charts/exp/exp__episode_reward_mean__timesteps_total__norm_fixed.png
💾 Salvo: ./exp_results/charts/happo/happo__episode_reward_mean__timesteps_total__norm_fixed.png
💾 Salvo: ./exp_results/charts/hatrpo/hatrpo__episode_reward_mean__timesteps_total__norm_fixed.png
💾 Salvo: ./exp_results/charts/ia2c/ia2c__episode_reward_mean__timesteps_total__norm_fixed.png
💾 Salvo: ./exp_results/charts/ippo/ippo__episode_reward_mean__timesteps_total__norm_fixed.png
💾 Salvo: ./exp_results/charts/itrpo/itrpo__episode_reward_mean__timesteps_total__norm_fixed.png
💾 Salvo: ./exp_results/charts/maa2c/maa2c__episode_reward_mean__timesteps_total__norm_fixed.png
💾 Salvo: ./exp_results/charts/mappo/mappo__episode_reward_mean__timesteps_total__norm_fixed.png
💾 Salvo: ./exp_results/charts/matrpo/matrpo__episode_reward_mean__timesteps_total__norm_fixed.png
✅ Concluído

# Codecarbon

In [8]:
# -*- coding: utf-8 -*-
"""
CodeCarbon charts — ONLY 'All Algos' with units
Entrada:  ./codecarbon/emissions.csv
Saída:    ./codecarbon/charts/all_algos/*.png
"""

import os
import warnings
from typing import List, Dict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore", category=FutureWarning)

# ========================= CONFIG =========================
CSV_PATH = "./codecarbon/emissions.csv"
OUT_DIR  = "./codecarbon/charts/all_algos"
os.makedirs(OUT_DIR, exist_ok=True)

# Métricas e unidades
UNITS = {
    "duration": "s",
    "emissions": "kg CO₂e",
    "emissions_rate": "kg CO₂e/s",
    "cpu_power": "W",
    "gpu_power": "W",
    "ram_power": "W",
    "cpu_energy": "kWh",
    "gpu_energy": "kWh",
    "ram_energy": "kWh",
    "energy_consumed": "kWh",
}

SEPARATED_METRICS = [
    "duration","emissions","emissions_rate",
    "cpu_power","gpu_power","ram_power",
    "cpu_energy","gpu_energy","ram_energy","energy_consumed",
]

COMPILED_GROUPS = {
    "powers": ["cpu_power","gpu_power","ram_power"],                      # W
    "energies": ["cpu_energy","gpu_energy","ram_energy","energy_consumed"],# kWh
    "emissions": ["emissions","emissions_rate"],                           # kg CO2e + kg CO2e/s
}

# ========================= HELPERS =========================
def ensure_numeric(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def parse_algorithm(project_name: str) -> str:
    """Espera 'framework:ALGO:run' -> 'algo'. Fallbacks razoáveis."""
    if not isinstance(project_name, str) or not project_name:
        return "unknown"
    parts = project_name.split(":")
    if len(parts) >= 2 and parts[1]:
        return parts[1].lower()
    if ":" in project_name:
        return project_name.split(":", 1)[-1].split(":")[0].lower() or "unknown"
    return project_name.lower()

def metric_exists(df: pd.DataFrame, metric: str) -> bool:
    return metric in df.columns and df[metric].notna().any()

def sci_fmt():
    return FuncFormatter(lambda x, pos: f"{x:.2g}")

def savefig(path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    print(f"💾 Salvo: {path}")

# ========================= LOAD =========================
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Arquivo não encontrado: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
df.rename(columns=str.strip, inplace=True)

# adiciona algorithm
if "project_name" not in df.columns:
    raise ValueError("Coluna 'project_name' ausente no CSV.")
df["algorithm"] = df["project_name"].apply(parse_algorithm)

# normaliza tipos
num_cols = [c for c in SEPARATED_METRICS if c in df.columns]
df = ensure_numeric(df, num_cols)

# ========================= SEPARATED: mean±std por algoritmo =========================
def bar_mean_std_by_algo(metric: str):
    if not metric_exists(df, metric):
        return
    g = df.groupby("algorithm")[metric].agg(["mean","std"]).dropna()
    if g.empty:
        return

    plt.figure(figsize=(12,7))
    bars = plt.bar(g.index, g["mean"], yerr=g["std"], capsize=5)
    plt.title(f"{metric} — mean ± std por algoritmo")
    unit = UNITS.get(metric, "")
    plt.ylabel(f"{metric} ({unit})" if unit else metric)
    plt.xticks(rotation=30, ha="right")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    savefig(os.path.join(OUT_DIR, f"all_algos__{metric}__bar_mean_std.png"))

for m in SEPARATED_METRICS:
    bar_mean_std_by_algo(m)

# ========================= COMPILED: multi-métricas no mesmo chart =========================
# Barras lado a lado por algoritmo (agrupadas) para powers/energies,
# e combinação barras + eixo secundário para emissions/emissions_rate.
def plot_grouped_bars(metrics: List[str], title: str, fname: str, secondary: str = None):
    # tabela mean por algoritmo x métrica
    have = [m for m in metrics if metric_exists(df, m)]
    if not have:
        return

    g = df.groupby("algorithm")[have].mean(numeric_only=True)
    g = g.dropna(how="all")
    if g.empty:
        return

    algos = g.index.tolist()
    x = np.arange(len(algos))
    width = 0.8 / len(have)  # largura por barra

    fig, ax1 = plt.subplots(figsize=(13,8))
    handles = []
    labels = []

    # Se tivermos 'secondary', ela vai no eixo 2 como marcadores sobre barras da primeira métrica
    sec_ax = None
    for i, m in enumerate(have):
        offset = (i - (len(have)-1)/2) * width
        if secondary and m == secondary:
            if sec_ax is None:
                sec_ax = ax1.twinx()
            # plota como linha com marcadores
            y = g[m].values
            h = sec_ax.plot(x, y, marker="o", linestyle="--", label=m)
            handles += h; labels += [m]
        else:
            y = g[m].values
            h = ax1.bar(x + offset, y, width=width, label=m)
            handles.append(h); labels.append(m)

    ax1.set_title(title)
    ax1.set_xticks(x, algos, rotation=30, ha="right")
    # Y labels com unidades (pega da primeira métrica não-secondary)
    main_metrics = [m for m in have if m != secondary]
    if main_metrics:
        unit_main = UNITS.get(main_metrics[0], "")
        ax1.set_ylabel(f"{main_metrics[0]} ({unit_main})" if unit_main else main_metrics[0])
    ax1.grid(axis="y", linestyle="--", alpha=0.4)

    if sec_ax and secondary:
        unit_sec = UNITS.get(secondary, "")
        sec_ax.set_ylabel(f"{secondary} ({unit_sec})" if unit_sec else secondary)
        sec_ax.yaxis.set_major_formatter(sci_fmt())

    # legenda combinada (barras + linha)
    if sec_ax:
        h2, l2 = sec_ax.get_legend_handles_labels()
        ax1.legend(handles + h2, labels + l2, loc="best")
    else:
        ax1.legend(loc="best")

    savefig(os.path.join(OUT_DIR, fname))

# Powers (W)
plot_grouped_bars(
    COMPILED_GROUPS["powers"],
    title="Powers — média por algoritmo",
    fname="compiled__powers_bar.png"
)

# Energies (kWh)
plot_grouped_bars(
    COMPILED_GROUPS["energies"],
    title="Energies — média por algoritmo",
    fname="compiled__energies_bar.png"
)

# Emissions (kg CO2e) + Emissions rate (kg CO2e/s) no eixo secundário
plot_grouped_bars(
    COMPILED_GROUPS["emissions"],
    title="Emissions & Emissions rate — média por algoritmo",
    fname="compiled__emissions_mix.png",
    secondary="emissions_rate"
)

print("✅ Concluído — gráficos salvos em", OUT_DIR)


💾 Salvo: ./codecarbon/charts/all_algos/all_algos__duration__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/all_algos__emissions__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/all_algos__emissions_rate__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/all_algos__cpu_power__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/all_algos__gpu_power__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/all_algos__ram_power__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/all_algos__cpu_energy__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/all_algos__gpu_energy__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/all_algos__ram_energy__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/all_algos__energy_consumed__bar_mean_std.png
💾 Salvo: ./codecarbon/charts/all_algos/compiled__powers_bar.png
💾 Salvo: ./codecarbon/charts/all_algos/compiled__energies_bar.png
💾 Salvo: ./codecarbon/charts/all_algos/compiled__emissions_mix.png
✅ Concluíd